# Financial Research Multi-Agent Pipeline — Full Walkthrough

This notebook walks through every phase of the pipeline step by step, displaying the output of each stage.

**Phases:**
1. Setup
2. Data Layer — Fundamentals, News, SEC Filings
3. RAG — Ingest & Retrieve
4. Agents — each specialist individually
5. Full Graph — parallel execution end to end
6. Final Report

---
## Phase 0 · Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from dotenv import load_dotenv
load_dotenv('../.env')

TICKER = 'AAPL'
QUERY  = "Analyze Apple's latest quarter"

print(f'Ticker : {TICKER}')
print(f'Query  : {QUERY}')

Ticker : AAPL
Query  : Analyze Apple's latest quarter


---
## Phase 1 · Data Layer

### 1a — Fundamentals (Yahoo Finance)

In [2]:
from fin_agents.data.fundamentals import get_fundamentals

f = get_fundamentals(TICKER)

print(f'Company    : {f.name} ({f.ticker})')
print(f'Sector     : {f.sector}')
print(f'Market Cap : ${f.market_cap:,.0f}')
print(f'P/E Ratio  : {f.pe_ratio}')
print(f'EPS        : {f.eps}')
print(f'Revenue TTM: ${f.revenue_ttm:,.0f}')
print(f'\nBusiness Summary (truncated):')
print(f.summary[:300] + '...' if f.summary else 'N/A')

Company    : Apple Inc. (AAPL)
Sector     : Technology
Market Cap : $3,909,280,858,112
P/E Ratio  : 33.710396
EPS        : 7.89
Revenue TTM: $435,617,005,568

Business Summary (truncated):
Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising Ai...


### 1b — News (NewsAPI)

In [3]:
from fin_agents.data.news import get_news

articles = get_news(TICKER, limit=5)

print(f'Found {len(articles)} articles\n')
for i, a in enumerate(articles, 1):
    print(f'[{i}] {a.title}')
    print(f'    Source   : {a.source}')
    print(f'    Published: {a.published_at.date()}')
    print(f'    URL      : {a.url}')
    print()

Found 5 articles

[1] 2 charts show why Magnificent 7 stocks are being loved again
    Source   : Yahoo Entertainment
    Published: 2026-04-20
    URL      : https://finance.yahoo.com/news/2-charts-show-why-magnificent-7-stocks-are-being-loved-again-151105348.html

[2] One Analyst Breaks From The Herd, Says Apple’s DRAM Buying Spree Won’t Crush Margins Despite 2.4 Exabyte Appetite
    Source   : Wccftech
    Published: 2026-04-20
    URL      : https://wccftech.com/goldman-sachs-breaks-from-wall-street-panic-says-apples-dram-hoarding-wont-crush-margins-despite-2-4-exabyte-appetite/

[3] Anker Japan、単ポート最大140W出力でMacBook Proの高速充電も可能なモバイルバッテリー「Anker Prime Power Bank (20100mAh, 220W)」と専用充電ベースのセットモデルを発売。
    Source   : Applech2.com
    Published: 2026-04-20
    URL      : https://applech2.com/archives/20260420-anker-prime-power-bank-20k-220w-with-base.html

[4] The S&P 500 has blown past 7,000 in an epic comeback rally. Here’s why it can keep going.
    Source   : MarketWatch
    Published

### 1c — SEC Filing (EDGAR)

In [4]:
from fin_agents.data.filings import latest_filing

filing = latest_filing(TICKER)

print(f'Ticker      : {filing.ticker}')
print(f'Filing Type : {filing.filing_type}')
print(f'Filing Date : {filing.filing_date}')
print(f'Text Length : {len(filing.text):,} characters')
print(f'\nFirst 500 chars of filing text:')
print('-' * 60)
print(filing.text[:500])

Ticker      : AAPL
Filing Type : 10-Q
Filing Date : 2026-01-30
Text Length : 66,160 characters

First 500 chars of filing text:
------------------------------------------------------------
aapl-20251227 false 2026 Q1 0000320193 --09-26 P1Y P1Y P1Y P1Y http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent http://fasb.org/us-gaap/2025#LongTermDebtNoncurrent P406D P403D xbrli:shares iso4217:USD iso4217:USD xbrli:shares xbrli:pure aapl:Customer aapl:Vendor 0000320193 2025-09-28 2025-12-27 0000320193 us-gaap:CommonStockMember 2025-09-28 2025-12-27 0000320193 aapl:A1.625NotesDue2026Member 2025-09-28 2025-12-27 0000320193 aapl:A2.000NotesDue2027Member 2025-09-28 2025-12-27 0000320193 aapl:


---
## Phase 2 · RAG — Ingest & Retrieve

### 2a — Ingest filing into ChromaDB

In [5]:
from fin_agents.rag.ingest import ingest_ticker, _collection

# Check if already ingested
col = _collection()
existing = col.get(where={'ticker': TICKER}, limit=1)

if existing['ids']:
    total = col.get(where={'ticker': TICKER})
    print(f'{TICKER} already ingested — {len(total["ids"])} chunks in DB')
else:
    print(f'Ingesting {TICKER}...')
    n = ingest_ticker(TICKER)
    print(f'Done — {n} chunks stored')

/Users/minaehab/Desktop/fin-research-agents/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 77997.38it/s]


AAPL already ingested — 51 chunks in DB


### 2b — Retrieve relevant chunks

In [6]:
from fin_agents.rag.retriever import retrieve

test_query = 'What are the main risks disclosed in the filing?'
chunks = retrieve(test_query, ticker=TICKER, k=3)

print(f'Query : {test_query}')
print(f'Chunks returned: {len(chunks)}\n')

for i, c in enumerate(chunks, 1):
    print(f'--- Chunk {i} (score: {c["score"]:.4f}) ---')
    print(f'Date : {c["meta"]["filing_date"]}  |  Type: {c["meta"]["filing_type"]}  |  Index: {c["meta"]["chunk_index"]}')
    print(c['text'][:300])
    print()

Query : What are the main risks disclosed in the filing?
Chunks returned: 3

--- Chunk 1 (score: 0.4324) ---
Date : 2026-01-30  |  Type: 10-Q  |  Index: 41
The preparation of financial statements and related disclosures in conformity with GAAP and the Company’s discussion and analysis of its financial condition and operating results require the Company’s management to make judgments, assumptions and estimates that affect the amounts reported. Note 1, “

--- Chunk 2 (score: 0.4507) ---
Date : 2026-01-30  |  Type: 10-Q  |  Index: 28
this Quarterly Report on Form 10-Q (“Form 10-Q”) contain forward-looking statements, within the meaning of the Private Securities Litigation Reform Act of 1995, that involve risks and uncertainties. Forward-looking statements provide current expectations of future events based on certain assumptions

--- Chunk 3 (score: 0.4853) ---
Date : 2026-01-30  |  Type: 10-Q  |  Index: 47
be materially adversely affected. Item 1A.    Risk Factors The Company’s business,

---
## Phase 3 · Individual Agents

### 3a — Fundamentals Agent

In [7]:
from fin_agents.agents.fundamentals_agent import run_fundamentals

state = {'query': QUERY, 'ticker': TICKER}
result = await run_fundamentals(state)
f = result['fundamentals']

print(f'Ticker     : {f.ticker}')
print(f'Market Cap : ${f.market_cap:,.0f}')
print(f'P/E        : {f.pe_ratio}')
print(f'\nLLM Commentary:')
print('-' * 60)
print(f.commentary)

⟳ fundamentals_agent  AAPL

✓ fundamentals_agent  AAPL  market_cap=$3,909,133,795,328

Ticker     : AAPL
Market Cap : $3,909,133,795,328
P/E        : 33.709126

LLM Commentary:
------------------------------------------------------------
Apple Inc. (AAPL) is trading at a P/E ratio of 33.71, indicating a premium valuation relative to the market. With an EPS of $7.89 and a market cap of nearly $3.91 trillion, the company's size and profitability are unparalleled in the technology sector. The revenue TTM of $435.62 billion suggests a stable top-line, but the valuation multiple implies investors are pricing in significant future growth. Notably, the current P/E ratio is elevated, which may pose a risk to investors if growth expectations are not met, highlighting the need for continued innovation and expansion into new markets to justify the premium valuation.


### 3b — News Agent

In [8]:
from fin_agents.agents.news_agent import run_news

state = {'query': QUERY, 'ticker': TICKER}
result = await run_news(state)

print(f'Overall Theme:')
print(result['news_summary'])
print(f'\nTagged Headlines ({len(result["news"])} articles):')
print('-' * 60)
for a in result['news']:
    print(f'[{a.sentiment:>8}]  {a.title}')

⟳ news_agent  AAPL

✓ news_agent  AAPL  5 articles tagged

Overall Theme:
The overall news theme is overwhelmingly positive, with multiple headlines indicating a surge in stock market performance and confidence in major tech companies. The S&P 500's comeback rally and the resurgence of interest in certain stocks are driving this optimism. This trend is also reflected in investment advice, with suggestions for building a portfolio around specific ETFs.

Tagged Headlines (5 articles):
------------------------------------------------------------
[positive]  2 charts show why Magnificent 7 stocks are being loved again
[positive]  One Analyst Breaks From The Herd, Says Apple’s DRAM Buying Spree Won’t Crush Margins Despite 2.4 Exabyte Appetite
[ neutral]  Anker Japan、単ポート最大140W出力でMacBook Proの高速充電も可能なモバイルバッテリー「Anker Prime Power Bank (20100mAh, 220W)」と専用充電ベースのセットモデルを発売。
[positive]  The S&P 500 has blown past 7,000 in an epic comeback rally. Here’s why it can keep going.
[positive]  3 ASX ETFs to build a portfolio around in 2026


### 3c — RAG Agent

In [9]:
from fin_agents.agents.rag_agent import run_rag

state = {'query': QUERY, 'ticker': TICKER}
result = await run_rag(state)
chunks = result['rag_chunks']

print(f'{len(chunks)} unique chunks retrieved\n')
for i, c in enumerate(chunks, 1):
    print(f'[{i}] Sub-query: {c["sub_query"]}')
    print(f'     Score   : {c["score"]:.4f}')
    print(f'     Preview : {c["text"][:150]}')
    print()

⟳ rag_agent  AAPL

✓ rag_agent  AAPL  3 sub-queries → 11 unique chunks

11 unique chunks retrieved

[1] Sub-query: What were Apple's revenue and net income in the latest quarter?
     Score   : 0.3055
     Preview : September. An additional week is included in the first fiscal quarter every five or six years to realign the Company’s fiscal quarters with calendar q

[2] Sub-query: What were Apple's revenue and net income in the latest quarter?
     Score   : 0.3117
     Preview : Services 30,013 26,340 Total net sales 143,756 124,300 Cost of sales: Products 67,478 59,447 Services 7,047 6,578 Total cost of sales 74,525 66,025 Gr

[3] Sub-query: What were Apple's revenue and net income in the latest quarter?
     Score   : 0.3167
     Preview : of net sales. As of December 27, 2025 and September 27, 2025, the Company had total deferred revenue of $ 14.3 billion and $ 13.7 billion, respectivel

[4] Sub-query: What were Apple's revenue and net income in the latest quarter?
     Score   : 0.3190
     Preview : December 27, 2025 December 28, 2024 Change Research 

---
## Phase 4 · Full Graph — Parallel Execution

In [10]:
import time
from fin_agents.agents.graph import build_graph

graph = build_graph()

initial_state = {
    'query': QUERY,
    'ticker': '',
    'fundamentals': None,
    'news': None,
    'news_summary': None,
    'rag_chunks': None,
    'final_report': None,
}

start = time.time()
final_state = await graph.ainvoke(initial_state)
elapsed = time.time() - start

print(f'Pipeline completed in {elapsed:.1f}s')
print(f'Ticker      : {final_state["ticker"]}')
print(f'Fundamentals: {final_state["fundamentals"].name if final_state["fundamentals"] else "None"}')
print(f'News items  : {len(final_state["news"] or [])}')
print(f'RAG chunks  : {len(final_state["rag_chunks"] or [])}')
print(f'Report len  : {len(final_state.get("final_report") or "")} chars')

⟳ orchestrator  query="Analyze Apple's latest quarter"

✓ orchestrator  ticker=AAPL

⟳ fundamentals_agent  AAPL

⟳ news_agent  AAPL

⟳ rag_agent  AAPL

...

✓ news_agent  AAPL  5 articles tagged

✓ fundamentals_agent  AAPL  market_cap=$3,908,619,206,656

⟳ synthesis_agent  AAPL

✓ synthesis_agent  AAPL  1835 chars

Pipeline completed in 3.5s
Ticker      : AAPL
Fundamentals: Apple Inc.
News items  : 5
RAG chunks  : 11
Report len  : 1835 chars


---
## Phase 5 · Final Report

In [11]:
from IPython.display import Markdown, display

display(Markdown(final_state['final_report']))

## Executive Summary
Apple Inc. (AAPL) is trading at a premium valuation with a P/E ratio of 33.70, driven by strong financial performance and growth prospects. The company's market capitalization of $3.91 trillion and revenue TTM of $435.6 billion underscore its scale and influence in the technology sector.

## Key Financials
Key financial metrics include:
- P/E ratio: 33.70
- EPS: $7.89
- Revenue TTM: $435.6 billion
- Market capitalization: $3.91 trillion
These metrics indicate a robust financial performance, justifying the premium valuation to some extent.

## Recent News
Recent news is overwhelmingly positive, with headlines indicating a surge in stock market performance and investor confidence. Notable positive news includes:
- The S&P 500's comeback rally
- Analyst predictions supporting bullish sentiment
- New product releases driving growth

## Filings Insights
SEC filing excerpts reveal:
- Total net sales of $143.756 billion, a 16% increase from the previous year
- Gross margin percentage of 48.2%, up from 46.9% in the previous year
- Research and development expenses grew by 32% to $10.887 billion
These insights suggest strong revenue growth and increasing investment in research and development.

## Risks
Potential risks include:
- High valuation multiple, which may not be sustainable if growth slows down
- Increasing research and development expenses, which could impact margins
- Dependence on new product introductions to drive growth

## Analyst Takeaway
The current valuation of Apple Inc. (AAPL) appears rich, but the company's strong financial performance, growth prospects, and dominant position in the technology sector may justify the premium. Investors should closely monitor future growth drivers, competitive advantages, and potential risks to determine if the current price is sustainable.